# Notebook 2 — Feature Engineering
Fetches weather → trains Isolation Forest on weather → merges all datasets → saves merged_dataset.csv + isolation_forest.pkl

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

DATA_DIR   = '../datasets/'
MODELS_DIR = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)
print('Imports done.')

Imports done.


In [2]:
d1 = pd.read_csv(DATA_DIR + 'dataset1_supply_chain_india.csv', parse_dates=['order_date'])
d3 = pd.read_csv(DATA_DIR + 'dataset3_news_sentiment_india.csv', parse_dates=['date'])
d4 = pd.read_csv(DATA_DIR + 'dataset4_supplier_risk_india.csv')
d5 = pd.read_csv(DATA_DIR + 'dataset5_historical_disruptions_india.csv', parse_dates=['event_date'])
print(f'D1:{d1.shape} D3:{d3.shape} D4:{d4.shape} D5:{d5.shape}')

D1:(2000, 15) D3:(600, 8) D4:(60, 10) D5:(1500, 10)


In [3]:
# City lat/lon for Open-Meteo
CITY_COORDS = {
    'Mumbai':    (19.08, 72.88), 'Delhi':     (28.61, 77.21),
    'Chennai':   (13.08, 80.27), 'Kolkata':   (22.57, 88.36),
    'Bangalore': (12.97, 77.59), 'Hyderabad': (17.38, 78.49),
    'Pune':      (18.52, 73.86), 'Ahmedabad': (23.03, 72.58),
    'Jaipur':    (26.91, 75.79), 'Lucknow':   (26.85, 80.95),
    'Surat':     (21.17, 72.83), 'Kanpur':    (26.46, 80.33),
    'Nagpur':    (21.15, 79.08), 'Indore':    (22.72, 75.86),
    'Bhopal':    (23.26, 77.41),
}
print(f'Coordinates defined for {len(CITY_COORDS)} cities.')

Coordinates defined for 15 cities.


In [4]:
# Fetch historical weather 2021-2024 from Open-Meteo API
# ~15 API calls, takes 1-2 minutes — do not interrupt

def fetch_weather_history(city, lat, lon):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': '2021-01-01', 'end_date': '2024-12-31',
        'daily': 'precipitation_sum,temperature_2m_max,wind_speed_10m_max',
        'timezone': 'Asia/Kolkata'
    }
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()['daily'])
    df.rename(columns={
        'time': 'date',
        'precipitation_sum': 'rainfall_mm',
        'temperature_2m_max': 'temperature_celsius',
        'wind_speed_10m_max': 'wind_speed_kmh'
    }, inplace=True)
    df['date'] = pd.to_datetime(df['date'])
    df['supplier_city'] = city
    return df

frames = []
for city, (lat, lon) in CITY_COORDS.items():
    print(f'  Fetching {city}...', end=' ')
    try:
        wf = fetch_weather_history(city, lat, lon)
        frames.append(wf)
        print(f'{len(wf)} rows')
    except Exception as e:
        print(f'ERROR: {e}')
    time.sleep(0.4)

weather_df = pd.concat(frames, ignore_index=True)
weather_df.fillna({'rainfall_mm': 0, 'wind_speed_kmh': 0}, inplace=True)
weather_df['temperature_celsius'] = weather_df['temperature_celsius'].fillna(
    weather_df.groupby('supplier_city')['temperature_celsius'].transform('median')
)
print(f'\nWeather dataframe: {weather_df.shape}')

  Fetching Mumbai... 1461 rows
  Fetching Delhi... 1461 rows
  Fetching Chennai... 1461 rows
  Fetching Kolkata... 1461 rows
  Fetching Bangalore... 1461 rows
  Fetching Hyderabad... 1461 rows
  Fetching Pune... 1461 rows
  Fetching Ahmedabad... 1461 rows
  Fetching Jaipur... 1461 rows
  Fetching Lucknow... 1461 rows
  Fetching Surat... 1461 rows
  Fetching Kanpur... 1461 rows
  Fetching Nagpur... 1461 rows
  Fetching Indore... 1461 rows
  Fetching Bhopal... 1461 rows

Weather dataframe: (21915, 5)


In [5]:
# Compute weather severity score (0/1/2)
def get_severity(row):
    if row['rainfall_mm'] > 50 or row['wind_speed_kmh'] > 60:
        return 2
    elif row['rainfall_mm'] > 15 or row['wind_speed_kmh'] > 35:
        return 1
    return 0

weather_df['severity_score'] = weather_df.apply(get_severity, axis=1)
print('Severity scores computed.')
print(weather_df['severity_score'].value_counts())

Severity scores computed.
severity_score
0    20267
1     1486
2      162
Name: count, dtype: int64


In [6]:
# ── ISOLATION FOREST on weather data ─────────────────────────────────────────
# Open-Meteo returns raw unlabelled numbers — no disruption label
# Isolation Forest learns what NORMAL weather looks like per city
# Then flags unusual weather as anomaly → weather_anomaly_score
# This score becomes a feature that feeds into XGBoost

WEATHER_FEATURES = ['rainfall_mm', 'temperature_celsius', 'wind_speed_kmh', 'severity_score']

# Train only on weather rows where severity = 0 (normal weather)
normal_weather = weather_df[weather_df['severity_score'] == 0][WEATHER_FEATURES]
print(f'Training Isolation Forest on {len(normal_weather)} normal weather records...')

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(normal_weather)

# Apply to ALL weather rows — anomaly score (-1=anomaly, 1=normal)
raw_pred = iso_forest.predict(weather_df[WEATHER_FEATURES])
weather_df['weather_anomaly_score'] = (raw_pred == -1).astype(int)  # 1=anomaly, 0=normal

print(f'Anomalies flagged: {weather_df["weather_anomaly_score"].sum()} out of {len(weather_df)}')

# Save Isolation Forest
joblib.dump(iso_forest, MODELS_DIR + 'isolation_forest.pkl')
print('isolation_forest.pkl saved.')

Training Isolation Forest on 20267 normal weather records...
Anomalies flagged: 2662 out of 21915
isolation_forest.pkl saved.


In [8]:
# Merge D1 + Weather on order_date + supplier_city
d1_merged = d1.merge(
    weather_df[['date','supplier_city','rainfall_mm','temperature_celsius',
                'wind_speed_kmh','severity_score','weather_anomaly_score']],
    left_on=['order_date','supplier_city'],
    right_on=['date','supplier_city'],
    how='left'
).drop(columns=['date'])

for col in ['rainfall_mm','temperature_celsius','wind_speed_kmh','severity_score','weather_anomaly_score']:
    d1_merged[col] = d1_merged[col].ffill().fillna(0)

print(f'After weather merge: {d1_merged.shape}')

After weather merge: (2000, 20)


In [9]:
# Merge D1 + D3 on order_date + city (avg sentiment per date+city)
d3_agg = d3.groupby(['date','affected_city']).agg(
    sentiment_score=('sentiment_score','mean'),
    disruption_signal=('disruption_signal','max')
).reset_index()

d1_merged = d1_merged.merge(
    d3_agg,
    left_on=['order_date','supplier_city'],
    right_on=['date','affected_city'],
    how='left'
).drop(columns=['date','affected_city'])

d1_merged['sentiment_score']   = d1_merged['sentiment_score'].fillna(0)
d1_merged['disruption_signal'] = d1_merged['disruption_signal'].fillna(0)
print(f'After sentiment merge: {d1_merged.shape}')

After sentiment merge: (2000, 22)


In [10]:
# Merge D1 + D4 on supplier_name
d4_cols = d4[['supplier_name','composite_risk_score','delivery_reliability_score',
               'regional_risk_index','strike_incidents_3yr','financial_stability_score']]
d1_merged = d1_merged.merge(d4_cols, on='supplier_name', how='left')

for col in ['composite_risk_score','delivery_reliability_score','regional_risk_index',
            'strike_incidents_3yr','financial_stability_score']:
    d1_merged[col] = d1_merged[col].fillna(d1_merged[col].median())

print(f'After supplier risk merge: {d1_merged.shape}')

After supplier risk merge: (8000, 27)


In [11]:
# Merge D1 + D5 — historical disruption count per city
d5_count = d5.groupby(['event_date','affected_city'])['disruption_occurred'].sum().reset_index()
d5_count.rename(columns={'disruption_occurred':'historical_disruption_count'}, inplace=True)

d1_merged = d1_merged.merge(
    d5_count,
    left_on=['order_date','supplier_city'],
    right_on=['event_date','affected_city'],
    how='left'
).drop(columns=['event_date','affected_city'])

d1_merged['historical_disruption_count'] = d1_merged['historical_disruption_count'].fillna(0)
print(f'After D5 merge: {d1_merged.shape}')

After D5 merge: (8000, 28)


In [12]:
# Engineered features — rolling delay averages and lag features per city
d1_merged = d1_merged.sort_values(['supplier_city','order_date']).reset_index(drop=True)

d1_merged['rolling_7d_delay']  = d1_merged.groupby('supplier_city')['delay_days'].transform(
    lambda x: x.rolling(7, min_periods=1).mean())
d1_merged['rolling_14d_delay'] = d1_merged.groupby('supplier_city')['delay_days'].transform(
    lambda x: x.rolling(14, min_periods=1).mean())
d1_merged['lag1_delay'] = d1_merged.groupby('supplier_city')['delay_days'].shift(1).fillna(0)
d1_merged['lag2_delay'] = d1_merged.groupby('supplier_city')['delay_days'].shift(2).fillna(0)

print('Rolling and lag features created.')

Rolling and lag features created.


In [13]:
# Encode categorical columns
le_transport = LabelEncoder()
le_product   = LabelEncoder()
le_city      = LabelEncoder()
le_dest      = LabelEncoder()

d1_merged['transport_mode_enc']    = le_transport.fit_transform(d1_merged['transport_mode'])
d1_merged['product_category_enc'] = le_product.fit_transform(d1_merged['product_category'])
d1_merged['supplier_city_enc']     = le_city.fit_transform(d1_merged['supplier_city'])
d1_merged['destination_city_enc']  = le_dest.fit_transform(d1_merged['destination_city'])

print('Categoricals encoded.')

# Save encoders for use in Notebook 5
joblib.dump(le_transport, MODELS_DIR + 'le_transport.pkl')
joblib.dump(le_product,   MODELS_DIR + 'le_product.pkl')
joblib.dump(le_city,      MODELS_DIR + 'le_city.pkl')
joblib.dump(le_dest,      MODELS_DIR + 'le_dest.pkl')
print('Label encoders saved.')

Categoricals encoded.
Label encoders saved.


In [14]:
# Final check and save
d1_merged.dropna(subset=['disruption_label'], inplace=True)

missing = d1_merged.isnull().sum()
missing = missing[missing > 0]
print('Remaining missing values:', dict(missing) if len(missing) else 'None')
print(f'Final merged dataset shape: {d1_merged.shape}')

d1_merged.to_csv(DATA_DIR + 'merged_dataset.csv', index=False)
print('\n✅ merged_dataset.csv saved.')
print('✅ isolation_forest.pkl saved.')
print('✅ Notebook 2 (Feature Engineering) complete.')

Remaining missing values: {'disruption_cause': 3964}
Final merged dataset shape: (8000, 36)

✅ merged_dataset.csv saved.
✅ isolation_forest.pkl saved.
✅ Notebook 2 (Feature Engineering) complete.
